### If necessary, install Cursor Extensions

1. From the View menu, select Extensions
2. Search for Python
3. Click on "Python" made by "ms-python" and select Install if not already installed
4. Search for Jupyter
5. Click on "Jupyter" made by "ms-toolsai" and select Install if not already installed


### Next Select the Kernel

Click on "Select Kernel" on the Top Right

Choose "Python Environments..."

Then choose the one that looks like `.venv (Python 3.12.x) .venv/bin/python` - it should be marked as "Recommended" and have a big star next to it.

Any problems with this? Head over to the troubleshooting.

### Note: you'll need to set the Kernel with every notebook..

In [3]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

# Connecting to OpenAI (or Ollama)

The next cell is where we load in the environment variables in your `.env` file and connect to OpenAI.  

If you'd like to use free Ollama instead, please see the README section "Free Alternative to Paid APIs", and if you're not sure how to do this, there's a full solution in the solutions folder (day1_with_ollama.ipynb).

## Troubleshooting if you have problems:

If you get a "Name Error" - have you run all cells from the top down? Head over to the Python Foundations guide for a bulletproof way to find and fix all Name Errors.

If that doesn't fix it, head over to the [troubleshooting](../setup/troubleshooting.ipynb) notebook for step by step code to identify the root cause and fix it!

Or, contact me! Message me or email ed@edwarddonner.com and we will get this to work.

Any concerns about API costs? See my notes in the README - costs should be minimal, and you can control it at every point. You can also use Ollama as a free alternative, which we discuss during Day 2.

In [4]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


# Let's make a quick call to a Frontier model to get started, as a preview!

In [5]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [7]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
response.choices[0].message.content

'Hello! It’s great to hear from you! How can I assist you today?'

## OK onwards with our first project

In [19]:
# Let's try out this utility

ed = fetch_website_contents("https://tnpsc.gov.in/home.aspx")
print(ed)

TNPSC - Tamil Nadu Public Service Commission

MAP
×
Sitemap
தமிழ்
For All examination related information for the year 2025 visit
Exam Dashboard
Menu
Home
About us
Vision and Mission
Role and Functions
Constitutional Provisions
The Commission
Historical perspective
Roll of Honour
TNPSC Regulations
Rules of Procedure
Office Orders
Grievance Redressal Officer
Annual Reports
Recruitment
Exam Dashboard
Syllabus
Scheme of Examination
Instructions to Applicants
Annual Planner
Selection Schedule
Archives (Examinations Before 2025)
Notifications
Results
Data Disclosure
Bulletin/E-Bulletin
Counselling
Question Papers / Answer Keys
Descriptive Type
Objective Type (Without Tentative Answer Keys)
Objective Type (With Answer Keys) / Previous year Question papers
Previous Year Question Papers
Equivalence
Government Orders
Judgements
Employee Corner
Departmental Examination
Apply Online
Fees Details
Notification
Download HALL Ticket
Syllabus
Books To Download
Pre.Year Question Papers
Answer Keys
Resu

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [9]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [10]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [11]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = openai.chat.completions.create(model="gpt-4o-mini",messages=messages)
response.choices[0].message.content

'2 + 2 equals 4.'

## And now let's build useful messages for GPT, using a function

In [12]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [13]:
# Try this out, and then try for a few more websites

messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula

## Time to bring it together - the API for OpenAI is very simple!

In [14]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4o-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [20]:
summarize("https://tnpsc.gov.in/home.aspx")

"Welcome to the TNPSC, where they clearly have a lot to say about every aspect of taming the wild world of Tamil Nadu's public service examinations. If you've ever dreamt of navigating a sea of notifications, syllabi, and exam schedules, then you’re in for a treat!\n\n### What's Cooking?\n- **Exams & Recruitment**: Upcoming recruitment for things like the Motor Vehicle Inspector position—because who doesn’t want to be up to their neck in vehicle regulations?\n- **Results & Notifications**: They seem to have a never-ending carousel of results and notifications that'll keep you on your toes. Just when you think you're done studying, boop! New exam details pop up!\n\n### In Summary\nIf you're not dizzy from all that paperwork and need to brush up on your public service exam game, check out their resources. Remember, when life gives you exams, make sure you're well-prepared—preferably with a side of sarcasm and a strong cup of coffee!"

In [16]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [24]:
display_summary("https://tnpsc.gov.in/home.aspx")

Welcome to the TNPSC website, where the Tamil Nadu Public Service Commission lays out everything you never knew you needed to know about taking exams that probably come with a side of anxiety. 

### Key Features:
- **Examinations Galore:** If you're itching to study for that big test in 2025, you might just find everything from syllabuses to previous years’ question papers. Because who doesn't love reliving the past, right?
- **Grievance Redressal:** Got a complaint? Don’t worry; there’s a whole section for that! 
- **Job Recruitment:** Offering a buffet of opportunities, but good luck getting through the maze of requirements.
- **Notifications:** Who needs surprises? Stay glued to their notifications for all the exam news that will likely keep you up at night.
- **Bulletin/E-Bulletin:** A fancy way of saying "here's all the important stuff you should know but probably won’t read."

In summary, if you're a fan of convoluted procedures, a plethora of forms, and a sprinkle of despair mixed with hope, the TNPSC website is just your cup of chai!

In [25]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown. Make sure you transalate the contents and provide the output in Tamil langugae strictly
"""

In [26]:
display_summary("https://www.google.com/finance/")

இந்த இணையதளத்தைப் பார்த்தால், உங்கள் பங்குகளைச் செங்குத்தாக செய்முறை செய்ய, வணிகச் செய்திகள், மற்றும் நேரடி விலைகளைத் திரட்டுவது போலவே, குறுநகைச்செய்யும் தளமாக இருக்கிறது என்று தெரிகிறது. 

தற்போது, சிறிது பங்கு அமைப்புகளின் நிலவரம்: 

- **AUTO**: -0.87% (சேலைப்பணார்போட்டிக்கு வேண்டாமா?)
- **ENERGY & UTILITIES**: -0.97% (எங்காடா அதன் நெருப்பு?)
- **FINANCIALS**: +0.03% (பணத்தில் உள்ள இது ஒரு குறியீட்டு மேலாண்மை!)
- **CONSUMER STAPLES**: -0.62% (சேமிக்கவும், நீர் இதற்கான கன்னத்தில் அல்ல!)
- **INDUSTRIALS**: +0.07% (எதைப் பார்த்தாலும் வெற்றி கிடைத்தது போல!)
- **INFORMATION TECHNOLOGY**: +2.32% (சமூக ஊடகங்கள் உங்களைப் பெற்றுள்ளன!)

கிளையுங்கள், நீங்கள் தொடர்ந்தால் இது உங்கள் பைணம் காயாதால், உங்கள் பவர்களைப் போல! 🌟

In [27]:
display_summary("https://anthropic.com")

இந்த வலைத்தளத்தின் சுருக்கம்:

ஆனால் ஆய்வானதே! ஆந்திரோபிக் என்னும் நிறுவனம், உலகத்தைக் காப்பாற்ற AIக்கும், பயன்கள் அதிகரிக்கவும் முயழுவதற்காக, பொது நன்மைக்கு Dedicated ஆன ஒரு நிறுவனம். நாளை யாரிடம் யாரும் பேசுவார்கள் என்றால், 'ஆய்வாளர்கள்' என்றவர்கள் இருக்கும்.

**செய்திகள்:**
1. **அக்டோபர் 7, 2026**: கிளாட் ஹைகு 5.5-ன் அறிமுகம்! இது முன்படியைப் போல் வேகமாகவும், திறமையுடனும் கொண்ட சிறிய மாதிரி.
2. **செப்டம்பர் 28, 2026**: கிளாட் சோனெட் 5.5! இது அருமை, அதனால் மாணவர்களின் அறிவு 30% மெருகூட்டப்படுகிறது.
3. **செப்டம்பர் 22, 2026**: கிளாட் ஓபஸ் 5.5! 40% குறைவான செலவில் பாரபட்சமாக அரிய மற்றும் அறிவுப் பணி சிறந்தது.

எப்படி இருந்தாலும், ஆராய்ச்சி, கொள்கைகள் மற்றும் பொது நன்மை மேல் அதிக கவனம் செலுத்த வேண்டியிருக்கிறது! 😏

In [28]:
display_summary("https://www.cricbuzz.com/")

**Cricbuzz** - உலகில் நோயெதிரүүлும் கிரிக்கெட் ரசிகர்களின் சுகாதார மையம்! 

இந்த இணையதளம் கிரிக்கெட் அட்டவணை, நேரடி கால் எண்கள், அசுர செய்திகள் மற்றும் வீடியோக்களுடன் கூடிய ஒரு மைதானம். இதன் மீது கமெண்ட் பண்ணவேண்டுமானால், இந்த சோம்பல் போட்டிகளில் எந்த ஒரு விளையாட்டானாலும் சைக்கிள் ஓட்டமாட்டும் நிலைமையில் உள்ளது. 

மேலும், 2026ல் நடக்கும் இந்திய குழுவின் நியூசிலாந்து பயணம், தென் ஆப்பிரிக்கா மற்றும் ஆஸ்திரேலியா ஆகிய அணிகளுக்கிடையிலான 3வது آز்மிற்கு எதிராக இருக்கும் 2026 T20 சவால் போன்றான நிறைய நிகழ்வுகள் உங்கள் எதிர்பார்ப்புகளுக்கு சூடெடுத்துதான் செய்தியுடன் கூடிய சிறு காய்கொள்! 

பார்த்தால், "உண்மையில், இன்னும் ஒருவர் படிக்க வேண்டும் என நினைத்து அதை தொடங்கி வரேன்!" 

## An extra exercise for those who enjoy web scraping

You may notice that if you try `display_summary("https://openai.com")` - it doesn't work! That's because OpenAI has a fancy website that uses Javascript. There are many ways around this that some of you might be familiar with. For example, Selenium is a hugely popular framework that runs a browser behind the scenes, renders the page, and allows you to query it. If you have experience with Selenium, Playwright or a similar browser automation framework, then feel free to implement an alternative version of the scraping functions in `scraper.py` (or in a separate module) using one of these tools. In the community-contributions folder, you'll find examples contributed by other students, including a Selenium implementation (thank you!)